# 04 — Evaluation Analysis

Deep-dive into model performance after training:
- Load checkpoint and run eval on val split
- Per question-type accuracy breakdown
- Visualise correct vs incorrect predictions
- Top-k answer probability distributions
- Error analysis by answer type

In [ ]:
import os, sys
os.chdir('..')
sys.path.insert(0, '.')

In [ ]:
import json
from pathlib import Path

import yaml
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import DataLoader
from transformers import BertTokenizer
from tqdm.notebook import tqdm

with open('configs/config.yaml') as f:
    cfg = yaml.safe_load(f)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 1. Load checkpoint, vocab, and val dataset

In [ ]:
from src.data.answer_vocab import AnswerVocab
from src.data.dataset import LocalVQADataset
from src.data.augmentations import get_val_transforms
from src.models.vqa_model import VQAModel
from src.utils.checkpoint import load_checkpoint

vocab = AnswerVocab.load(cfg['paths']['vocab_path'])
tokenizer = BertTokenizer.from_pretrained(cfg['model']['text_encoder'])

ckpt_dir = Path(cfg['paths']['checkpoint_dir'])
ckpts = sorted(ckpt_dir.glob('checkpoint_epoch*.pt'))
if not ckpts:
    raise FileNotFoundError('No checkpoints found. Train the model first.')
latest_ckpt = ckpts[-1]
print(f'Loading checkpoint: {latest_ckpt}')

d = cfg['data']
val_ds = LocalVQADataset(
    annotations_path=d['annotations_val'],
    questions_path=d['questions_val'],
    images_dir=d['images_val'],
    split='val',
    vocab=vocab,
    image_transform=get_val_transforms(d['image_size']),
    tokenizer=tokenizer,
    max_question_length=d['max_question_length'],
)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=4)
print(f'Val samples: {len(val_ds):,}')

In [ ]:
model = VQAModel(
    vision_backbone=cfg['model']['vision_backbone'],
    text_encoder=cfg['model']['text_encoder'],
    hidden_dim=cfg['model']['hidden_dim'],
    num_heads=cfg['model']['num_heads'],
    fusion_layers=cfg['model']['fusion_layers'],
    num_answer_classes=len(vocab),
).to(device)
load_checkpoint(latest_ckpt, model, device=device)
model.eval()
print('Model loaded.')

## 2. Run evaluation

In [ ]:
from src.evaluation.vqa_eval import VQAEvaluator
from src.evaluation.metrics import top_k_accuracy

evaluator = VQAEvaluator(vocab)
all_preds, all_labels, topk_acc = [], [], []

with torch.no_grad():
    for batch in tqdm(val_loader, desc='Evaluating'):
        logits = model(
            pixel_values=batch['image'].to(device),
            input_ids=batch['input_ids'].to(device),
            attention_mask=batch['attention_mask'].to(device),
        )
        evaluator.process_batch(batch['question_id'], logits, batch['soft_scores'])
        topk_acc.append(top_k_accuracy(logits.cpu(), batch['soft_scores'], k=cfg['evaluation']['top_k_answers']))
        all_preds.extend(logits.argmax(-1).cpu().tolist())
        all_labels.extend(batch['label'].tolist())

results = evaluator.summarise()
results['top_k_accuracy'] = sum(topk_acc) / len(topk_acc)
print(json.dumps(results, indent=2))

## 3. Per question-type accuracy

In [ ]:
import json as _json
from src.evaluation.metrics import per_type_accuracy

ann_val = _json.loads(Path(cfg['data']['annotations_val']).read_text())
q_val   = _json.loads(Path(cfg['data']['questions_val']).read_text())

# Build question_id -> question_type mapping
qtype_map = {a['question_id']: a['question_type'] for a in ann_val['annotations']}

# per_type_accuracy expects predictions and soft scores grouped by type
# (placeholder — wire up to your evaluator's per_type breakdown)
print('Per-type accuracy breakdown: implement via evaluator.per_type_scores()')

## 4. Visualise correct and incorrect predictions

In [ ]:
from src.evaluation.visualisation import visualise_predictions

# visualise_predictions(model, val_ds, vocab, device, n=8, save_path='outputs/eval_plots/predictions.png')
print('Call visualise_predictions() to generate a prediction grid.')

## 5. Top-k answer probability distribution

In [ ]:
import random
from PIL import Image as PILImage
from src.data.augmentations import get_val_transforms

transform = get_val_transforms(cfg['data']['image_size'])
sample = random.choice(val_ds.samples)  # (question_id, image_id, question, answers)
qid, image_id, question, answers = sample
img_path = Path(cfg['data']['images_val']) / f"COCO_val2014_{image_id:012d}.jpg"

pixel_values = transform(PILImage.open(img_path).convert('RGB')).unsqueeze(0).to(device)
enc = tokenizer(question, return_tensors='pt', padding='max_length', truncation=True,
                max_length=cfg['data']['max_question_length'])

with torch.no_grad():
    probs, indices = model.predict(
        pixel_values, enc['input_ids'].to(device), enc['attention_mask'].to(device),
        top_k=cfg['evaluation']['top_k_answers'],
    )

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(PILImage.open(img_path).convert('RGB'))
axes[0].set_title(f'Q: {question}', fontsize=9, wrap=True)
axes[0].axis('off')

top_ans   = [vocab.idx_to_answer(i.item()) for i in indices[0]]
top_probs = [p.item() * 100 for p in probs[0]]
axes[1].barh(top_ans[::-1], top_probs[::-1])
axes[1].set_xlabel('Probability (%)')
axes[1].set_title('Top-k predictions')

plt.tight_layout()
Path('outputs/eval_plots').mkdir(parents=True, exist_ok=True)
plt.savefig('outputs/eval_plots/topk_example.png', dpi=150)
plt.show()